In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests as rqs

In [0]:
%skip
url = ""
path = ""
response = rqs.get(url)
status = response.status_code

try:
    if status == 200:
        with open(path, "wb") as f:
            f.write(response.content)
            print("file saved")
    else:
        print("file not saved")
except:
    print("error in file", e)


In [0]:
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw/space_debris_raw.csv"

df = pd.read_csv(path)
df

In [0]:
df.head(4)

In [0]:
df.tail(4)

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df.describe()

In [0]:
df.columns

In [0]:
rename_mapping={
        "OBJECT_NAME": "ObjectName",
        "OBJECT_ID": "ObjectID",
        "NORAD_CAT_ID": "CatalogID",
        "OBJECT_TYPE": "ObjectType",
        "OPS_STATUS_CODE": "OperationalStatus",
        "OWNER": "Owner",
        "LAUNCH_DATE": "LaunchDate",
        "LAUNCH_SITE": "LaunchSite",
        "DECAY_DATE": "DecayDate",
        "PERIOD": "OrbitalPeriodMin",
        "INCLINATION": "InclinationDegrees",
        "APOGEE": "MaxAltitudeKM",
        "PERIGEE": "MinAltitudeKM",
        "RCS": "RadarSizeSQM",
        "DATA_STATUS_CODE": "DataStatus",
        "ORBIT_CENTER": "OrbitCenter",
        "ORBIT_TYPE": "OrbitState",
    }

df= df.rename(columns=rename_mapping)
df

In [0]:
print("before")

for i in df.columns:
    if df[i].dtype == "object":
        wht = (df[i].notna()) & (df[i] != df[i].str.strip())
        cont = wht.sum()
        if cont > 0:
            print(f"{i}:{cont}")
        else:
            print(f"{i}:{cont}")
print("\nremoving....\n")
for i in df.columns:
    if df[i].dtype == "object":
        df[i] = df[i].str.strip()

print("after")
for i in df.columns:
    if df[i].dtype == "object":
        wht = (df[i].notna()) & (df[i] != df[i].str.strip())
        cont = wht.sum()
        if cont > 0:
            print(f"{i}:{cont}")
        else:
            print(f"{i}:{cont}")


In [0]:
# find the outlier

num = df.select_dtypes(include=["number"])

q1 = num.quantile(0.25)
q3 = num.quantile(0.75)
iqr = q3-q1

lowerbound = q1 - 1.5 * iqr
uperbound = q3 + 1.5 * iqr

outlier = (num < lowerbound) |(num > uperbound)

cont = outlier.sum()
print(f"{num}: {cont}")

# if you want to see row

row = df[outlier.any(axis=1)]
print(row)


plt.figure(figsize=(18,12))

plt.subplot(2,3,1)
sns.boxplot(df["OrbitalPeriodMin"])
plt.title("OrbitalPeriodMin")

plt.subplot(2,3,2)
sns.boxplot(df["MinAltitudeKM"])
plt.title("MinAltitudeKM")

plt.subplot(2,3,3)
sns.boxplot(df["MaxAltitudeKM"])
plt.title("MaxAltitudeKM")

plt.subplot(2,3,4)
sns.boxplot(df["InclinationDegrees"])
plt.title("InclinationDegrees")

plt.subplot(2,3,5)
sns.boxplot(df["RadarSizeSQM"])
plt.title("RadarSizeSQM")

plt.xlabel("num")
plt.ylabel("num")
plt.show()

In [0]:
# find the duplicates in data

df.duplicated().sum()



In [0]:
num = df.isnull().sum()
prt = (num/len(df)*100).round(2)

dic = pd.DataFrame({"Numbers": num, "Percentage": prt})

dic

In [0]:
df["OrbitCenter"].unique()

In [0]:
col = ['OrbitalPeriodMin', 
       'InclinationDegrees', 
       'MaxAltitudeKM',
       'MinAltitudeKM']

uni = df["OrbitCenter"].unique()

for i in uni:
    flt = df["OrbitCenter"] == i
    for j in col:
        df.loc[flt, j] = df.loc[flt, j].fillna(df[flt].groupby("ObjectType")[j].transform("mean"))
df.isnull().sum()

operation_status_code: Operational status indicator:
+: Operational/Active.
-: Non-operational/Inactive.
P : Partially Operational / Standby
D: Decayed (re-entered Earth's atmosphere).
NaN: Unknown or unassigned status.

In [0]:
df["OperationalStatus"] = df["OperationalStatus"].fillna("Unknown")

df["OperationalStatus"].isnull().sum()

### Data Standardization    

In [0]:
dic = {"+": "Active", 
"-": "Inactive",
"P" : "standby",
"D": "Decayed"}

df["OperationalStatus"] = df["OperationalStatus"].replace(dic)

## binning

In [0]:
degree_bins = [0, 20, 80, 100, 180]
labels = ["Equatorial", "Mid-Inclination", "Polar / SSO", "Retrograde"]

df["InclinationDegreesbins"] = pd.cut(df["InclinationDegrees"], bins=degree_bins, labels=labels)

df